# Restock Radar — Next-Week Grocery Purchase Ranking: reference solution

**Approach:** per-user candidate generation + LightGBM LambdaRank, trained on a
*temporal* holdout (features from weeks ≤ 10, labels = week-11 purchases),
then refit with the feature window advanced one week to predict evaluation
week 12.

Key data facts this solution is built around (verified in the audit below):

1. ~1.5% of log rows are **replayed duplicates** — deduplicate before any counting.
2. **Promotions inflate views far more than purchases** (banner traffic
   converts poorly), so views are a biased popularity signal; purchases are
   the trustworthy event.
3. **~⅔ of next-week purchases are repeats** with per-item restock rhythms —
   "how overdue is this item for this user" (dueness) beats raw recency.
4. A slice of eval-week purchases are **newly launched items** with almost no
   interaction history — content/promo features must cover them.


In [1]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

T0 = time.time()
SEED = 42
K = 20
WEEK0 = pd.Timestamp("2025-01-06")   # Monday of week 0, per dataset docs
DATA = "./dataset/public"

log = pd.read_csv(f"{DATA}/train.csv")
items = pd.read_csv(f"{DATA}/items.csv")
users = pd.read_csv(f"{DATA}/users.csv")
promos = pd.read_csv(f"{DATA}/promotions.csv")
test = pd.read_csv(f"{DATA}/test.csv")

n0 = len(log)
log = log.drop_duplicates()          # remove log-replay duplicates
print(f"removed {n0 - len(log):,} replayed rows ({(n0 - len(log)) / n0:.2%})")

log["ts"] = pd.to_datetime(log["timestamp"])
log["week"] = ((log.ts - WEEK0).dt.days // 7).astype(int)
promos["week"] = ((pd.to_datetime(promos.week_start) - WEEK0).dt.days // 7).astype(int)
items["launch_week"] = ((pd.to_datetime(items.launch_date) - WEEK0).dt.days // 7).astype(int)
cat_of = items.set_index("item_id").category
price_of = items.set_index("item_id").unit_price

purch = log[log.event_type == "purchase"]
views = log[log.event_type == "view"]
carts = log[log.event_type == "add_to_cart"]
print(log.event_type.value_counts().to_string())

removed 35,491 replayed rows (1.48%)


event_type
view           1736213
add_to_cart     366518
purchase        263325


## Data audit: verify the traps before modeling

Promotion weeks multiply an item's views by ~2.5× but its purchases by only
~1.35× — the conversion rate on promo weeks is roughly **half** of normal.
Any feature built on raw view counts therefore systematically overrates
promoted items. We also measure how much of a week's purchasing is repeat
behaviour, which motivates the dueness features.

In [2]:
pv = views.groupby(["week", "item_id"]).size().rename("v")
pp = purch.groupby(["week", "item_id"]).size().rename("p")
wk = pd.concat([pv, pp], axis=1).fillna(0).reset_index()
pset = set(zip(promos.week, promos.item_id))
wk["promo"] = [(w, i) in pset for w, i in zip(wk.week, wk.item_id)]
g = wk.groupby("promo")[["v", "p"]].mean()
g["conv"] = g.p / g.v
print("per item-week means:")
print(g.round(3).to_string())
print(f"view lift x{g.loc[True,'v']/g.loc[False,'v']:.2f}, "
      f"purchase lift x{g.loc[True,'p']/g.loc[False,'p']:.2f}")

w11 = purch[purch.week == 11][["user_id", "item_id"]].drop_duplicates()
before = purch[purch.week < 11][["user_id", "item_id"]].drop_duplicates()
rep_share = w11.merge(before.assign(s=1), how="left",
                      on=["user_id", "item_id"]).s.notna().mean()
print(f"repeat share of week-11 purchases: {rep_share:.1%}")

per item-week means:
             v       p   conv
promo                        
False   61.787   9.923  0.161
True   157.319  13.416  0.085
view lift x2.55, purchase lift x1.35
repeat share of week-11 purchases: 67.5%


## Candidates + features

For each target user we score a few hundred candidates: everything they ever
bought, co-purchase neighbours of their items, the top recent sellers in
their favourite categories, newly launched and next-week-promoted items in
those categories, and the global top sellers.

Features per (user, item):

- **Repeat/habit:** purchase counts (total & recent), weeks since last
  purchase, and **dueness** = weeks-since-last ÷ that user-item's typical
  inter-purchase gap (fallback: category median gap). A weekly staple 1 week
  after purchase is *due* (≈1); a 6-week household item 1 week after
  purchase is not (≈0.17) — raw recency confuses exactly these two cases.
- **Item demand:** recent *purchase* counts and smoothed view→purchase
  conversion (promo-robust), raw view counts left in so the model can learn
  their bias, next-week promo flag, launch recency.
- **Fit:** user's purchase share in the item's category, price ratio vs the
  user's median spend in that category, max co-purchase similarity to the
  user's basket history.

In [3]:
def build_frame(end_week, target_users, label_week=None):
    """Candidates + features from weeks <= end_week for target_users."""
    P = purch[purch.week <= end_week]
    V = views[views.week <= end_week]
    C = carts[carts.week <= end_week]
    r0 = end_week - 3                                  # recent = last 4 weeks

    ui = P.groupby(["user_id", "item_id"]).agg(
        n_purch=("week", "size"), last_w=("week", "max"),
        first_w=("week", "min"), n_weeks=("week", "nunique")).reset_index()
    ui["weeks_since"] = end_week + 1 - ui.last_w
    span = (ui.last_w - ui.first_w).where(ui.n_weeks > 1)
    ui["gap"] = (span / (ui.n_weeks - 1).clip(lower=1)).fillna(0)
    tmp = ui.merge(items[["item_id", "category"]], on="item_id")
    cat_gap = tmp[tmp.gap > 0].groupby("category").gap.median()
    ui = ui.merge(items[["item_id", "category"]], on="item_id")
    ui["gap_f"] = ui.gap.where(ui.gap > 0, ui.category.map(cat_gap)) \
        .fillna(2.0).clip(lower=0.5)
    ui["dueness"] = (ui.weeks_since / ui.gap_f).clip(0, 4)

    uir = P[P.week >= r0].groupby(["user_id", "item_id"]).size().rename("n_purch_r4")
    uiv = V[V.week >= r0].groupby(["user_id", "item_id"]).size().rename("n_views_r4")
    uic = C[C.week >= r0].groupby(["user_id", "item_id"]).size().rename("n_carts_r4")

    ip = P[P.week >= r0].groupby("item_id").size().rename("item_purch_r4")
    iv = V[V.week >= r0].groupby("item_id").size().rename("item_views_r4")
    istat = pd.concat([ip, iv], axis=1).fillna(0)
    istat["item_conv_r4"] = (istat.item_purch_r4 + 1) / (istat.item_views_r4 + 20)
    promo_next = set(promos[promos.week == end_week + 1].item_id)

    pc = P.merge(items[["item_id", "category"]], on="item_id")
    ucat = pc.groupby(["user_id", "category"]).size().rename("u_cat_n").reset_index()
    ucat["u_cat_share"] = ucat.u_cat_n / ucat.groupby("user_id").u_cat_n.transform("sum")
    upr = pc.assign(price=pc.item_id.map(price_of)).groupby(
        ["user_id", "category"]).price.median().rename("u_cat_price")

    # item-item normalized co-purchase matrix
    uidx = {u: i for i, u in enumerate(P.user_id.unique())}
    iidx = {it: i for i, it in enumerate(items.item_id)}
    X = np.zeros((len(uidx), len(items)), dtype=np.float32)
    pu = P[["user_id", "item_id"]].drop_duplicates()
    X[pu.user_id.map(uidx), pu.item_id.map(iidx)] = 1.0
    co = X.T @ X
    popv = np.diag(co).copy()
    np.fill_diagonal(co, 0)
    S = co / np.sqrt(np.outer(popv + 5, popv + 5))
    top_partners = np.argsort(-S, axis=1)[:, :10]
    inv_item = items.item_id.to_numpy()

    pop_top = ip.sort_values(ascending=False).head(40).index.tolist()
    new_items = items[items.launch_week >= end_week - 2]
    cat_top = (pc[pc.week >= r0].groupby(["category", "item_id"]).size()
               .rename("n").reset_index().sort_values("n", ascending=False)
               .groupby("category").head(15))
    cat_top_map = cat_top.groupby("category").item_id.apply(list).to_dict()
    new_by_cat = new_items.groupby("category").item_id.apply(list).to_dict()
    promo_by_cat = (items[items.item_id.isin(promo_next)]
                    .groupby("category").item_id.apply(list).to_dict())
    mine_map = ui.groupby("user_id").item_id.apply(list).to_dict()
    ucat_top = (ucat.sort_values("u_cat_share", ascending=False)
                .groupby("user_id").category.apply(lambda s: s.head(4).tolist())
                .to_dict())

    rows_u, rows_i = [], []
    for u in target_users:
        cands = set(mine_map.get(u, []))
        for it in mine_map.get(u, [])[:60]:
            for j in top_partners[iidx[it]]:
                cands.add(inv_item[j])
        for c in ucat_top.get(u, []):
            cands.update(cat_top_map.get(c, []))
            cands.update(new_by_cat.get(c, [])[:10])
            cands.update(promo_by_cat.get(c, [])[:10])
        cands.update(pop_top)
        cands = sorted(cands)
        rows_u.extend([u] * len(cands))
        rows_i.extend(cands)

    F = pd.DataFrame({"user_id": rows_u, "item_id": rows_i})
    F = F.merge(ui.drop(columns=["category"]), on=["user_id", "item_id"], how="left")
    F = F.merge(uir, on=["user_id", "item_id"], how="left")
    F = F.merge(uiv, on=["user_id", "item_id"], how="left")
    F = F.merge(uic, on=["user_id", "item_id"], how="left")
    F = F.merge(istat, on="item_id", how="left")
    F["category"] = F.item_id.map(cat_of)
    F = F.merge(ucat[["user_id", "category", "u_cat_share"]],
                on=["user_id", "category"], how="left")
    F = F.merge(upr.reset_index(), on=["user_id", "category"], how="left")
    F["price"] = F.item_id.map(price_of)
    F["price_ratio"] = (F.price / F.u_cat_price).fillna(1.0).clip(0.2, 5)
    F["promo_next"] = F.item_id.isin(promo_next).astype(int)
    F["launch_week"] = F.item_id.map(items.set_index("item_id").launch_week)
    F["is_new"] = (F.launch_week >= end_week - 2).astype(int)

    smax = np.zeros(len(F), dtype=np.float32)
    fi = F.item_id.map(iidx).to_numpy()
    for u, grp in F.groupby("user_id", sort=False).indices.items():
        mine = [iidx[it] for it in mine_map.get(u, [])]
        if mine:
            smax[grp] = S[np.ix_(mine, fi[grp])].max(axis=0)
    F["cooc"] = smax

    fill0 = ["n_purch", "n_weeks", "n_purch_r4", "n_views_r4", "n_carts_r4",
             "item_purch_r4", "item_views_r4", "item_conv_r4", "u_cat_share"]
    F[fill0] = F[fill0].fillna(0)
    F["weeks_since"] = F.weeks_since.fillna(99)
    F["dueness"] = F.dueness.fillna(0)
    F["carted_not_bought"] = ((F.n_carts_r4 > 0) & (F.n_purch_r4 == 0)).astype(int)

    if label_week is not None:
        lab = purch[purch.week == label_week][["user_id", "item_id"]] \
            .drop_duplicates().assign(rel=1)
        F = F.merge(lab, on=["user_id", "item_id"], how="left")
        F["rel"] = F.rel.fillna(0).astype(int)
    return F


FEATS = ["n_purch", "n_weeks", "n_purch_r4", "weeks_since", "dueness",
         "n_views_r4", "n_carts_r4", "carted_not_bought", "item_purch_r4",
         "item_views_r4", "item_conv_r4", "u_cat_share", "price_ratio",
         "promo_next", "is_new", "cooc"]


def ndcg_at_k(sub, ans, k=K):
    ans = ans[["user_id", "item_id"]].drop_duplicates().assign(rel=1)
    m = sub.merge(ans, on=["user_id", "item_id"], how="left")
    m["rel"] = m.rel.fillna(0)
    m["gain"] = m.rel / np.log2(m["rank"] + 1.0)
    dcg = m.groupby("user_id").gain.sum()
    pos = ans.groupby("user_id").size().clip(upper=k)
    disc = 1 / np.log2(np.arange(1, k + 1) + 1.0)
    cum = np.concatenate([[0.0], np.cumsum(disc)])
    idcg = pd.Series(cum[pos.to_numpy()], index=pos.index)
    return float((dcg / idcg).reindex(sub.user_id.unique()).fillna(0).mean())


def topk_from_scores(F, scores, k=K):
    F = F.assign(score=scores)
    F = F.sort_values(["user_id", "score", "item_id"],
                      ascending=[True, False, True])
    F = F.groupby("user_id").head(k).copy()
    F["rank"] = F.groupby("user_id").cumcount() + 1
    return F[["user_id", "rank", "item_id"]]


def fit_ranker(tr):
    tr = tr.sort_values("user_id", kind="mergesort")
    grp = tr.groupby("user_id", sort=False).size().to_numpy()
    mdl = lgb.LGBMRanker(objective="lambdarank", n_estimators=400,
                         learning_rate=0.05, num_leaves=63,
                         min_child_samples=40, random_state=SEED,
                         deterministic=True, force_row_wise=True, verbose=-1)
    mdl.fit(tr[FEATS], tr.rel, group=grp)
    return mdl

## Temporal validation on week 11

The logs are dense repeat-purchase streams: a random row split would put the
same user's near-identical habit rows in both folds and grossly overstate
skill. Instead we reconstruct the task one week earlier — features from
weeks ≤ 10, labels = week-11 purchases, with validation users mirroring the
official target definition (≥3 recent purchases + active in the label week)
— and check we beat the strong repeat baseline and the popularity baseline
under the exact evaluation metric.

In [4]:
val_week = 11
rec = purch[(purch.week >= val_week - 4) & (purch.week < val_week)]
rc = rec[["user_id", "item_id", "week"]].drop_duplicates().groupby("user_id").size()
val_buyers = set(purch[purch.week == val_week].user_id)
val_users = sorted(set(rc[rc >= 3].index) & val_buyers)
rng = np.random.default_rng(SEED)
val_users = sorted(rng.choice(np.array(val_users),
                              size=min(2500, len(val_users)),
                              replace=False).tolist())
TR = build_frame(val_week - 1, val_users, label_week=val_week)
pos_total = purch[(purch.week == val_week) &
                  (purch.user_id.isin(set(val_users)))] \
    .groupby(["user_id", "item_id"]).ngroups
print(f"[{time.time()-T0:.0f}s] frame {len(TR):,} rows, "
      f"{TR.rel.mean():.2%} positive, candidate recall "
      f"{TR.groupby('user_id').rel.sum().sum() / pos_total:.1%}")

cut = int(len(val_users) * 0.8)
tr_u, va_u = set(val_users[:cut]), set(val_users[cut:])
tr, va = TR[TR.user_id.isin(tr_u)], TR[TR.user_id.isin(va_u)]
mdl = fit_ranker(tr)

va_ans = purch[purch.week == val_week][["user_id", "item_id"]]
va_ans = va_ans[va_ans.user_id.isin(va_u)]
print(f"[{time.time()-T0:.0f}s] validation NDCG@20 (week 11): "
      f"{ndcg_at_k(topk_from_scores(va, mdl.predict(va[FEATS])), va_ans):.4f}")
print("  repeat-count baseline:",
      round(ndcg_at_k(topk_from_scores(
          va, (va.n_purch * 10 - va.weeks_since).to_numpy()), va_ans), 4))
print("  popularity baseline:  ",
      round(ndcg_at_k(topk_from_scores(
          va, va.item_purch_r4.to_numpy()), va_ans), 4))
imp = sorted(zip(FEATS, mdl.feature_importances_), key=lambda t: -t[1])
print("top features:", ", ".join(f"{f}({g})" for f, g in imp[:6]))

[5s] frame 483,500 rows, 3.05% positive, candidate recall 86.3%


[12s] validation NDCG@20 (week 11): 0.5222
  repeat-count baseline: 0.3828
  popularity baseline:   0.0283
top features: u_cat_share(4235), cooc(3984), price_ratio(3505), item_conv_r4(2896), item_views_r4(2156), item_purch_r4(2096)


## Final model: refit and predict evaluation week 12

Refit on the full week-11 training frame, advance the feature window to
weeks ≤ 11, score candidates for the 1,600 official target users, and write
the ranked top-20 per user in the required `user_id,rank,item_id` format.

In [5]:
import os
mdl_full = fit_ranker(TR)
targets = test.user_id.tolist()
FF = build_frame(11, targets, label_week=None)
sub = topk_from_scores(FF, mdl_full.predict(FF[FEATS]))

assert sub.user_id.nunique() == len(targets)
assert (sub.groupby("user_id").size() == K).all()
assert not sub.duplicated(["user_id", "item_id"]).any()

os.makedirs("./working", exist_ok=True)
sub.to_csv("./working/submission.csv", index=False)
print(f"[{time.time()-T0:.0f}s] wrote ./working/submission.csv "
      f"({len(sub):,} rows, {sub.user_id.nunique()} users)")

[24s] wrote ./working/submission.csv (32,000 rows, 1600 users)


## Remarks

- Week-11 validation lands around **0.52 NDCG@20**, vs ≈0.38 for the
  repeat-count heuristic and ≈0.03 for global popularity — personalization,
  restock timing, promo-robust demand signals and cold-item coverage each
  add real headroom.
- A perfect score is unreachable: a slice of next-week purchases are
  first-time discoveries with little predictive trace, and brand-new items
  have almost no history. Chasing beyond ~0.6–0.7 likely means overfitting
  the public weeks.
- Everything is seeded and deterministic; the notebook runs end-to-end in
  well under the time budget on CPU only.